# ELISa on an A100: four changing binary-star proposals

Select an **A100 GPU** in Colab, then **Run all**. The last cell downloads a ZIP with the measurements and logs. Disconnect and delete the runtime when finished. First installation downloads approximately 2.65 GB of wheels. A retry on the same runtime reuses the verified checkout and installed packages, writes a separate evidence ZIP, and records any CUDA library override.

This repeats the m022 RTX 3080 Ti workload with the same circular detached synthetic binaries, two field resolutions, curved Roche surfaces, limb darkening, irradiation and area-integrated intrinsic temperatures. Each of three independent processes evaluates a first batch of four and a second batch of four changed proposals. Each returns complete curves, original mapped predictions, log densities, adaptive records and resolution checks. One warm sample per backend is a preliminary measurement.

**What the result means.** We measure serial finite-model CPU, JAX CPU and GPU on the Colab host, and compare the GPU process with the archived RTX measurement. Cross-host times include differences in host CPU, compiler and PCIe configuration. The accuracy test stays at nominal uncertainty 10⁻⁴, distance ≤ 0.1 and log-likelihood bound ≤ 0.01. The fine field/boundary test on the original host failed. This run measures hardware and implementation parity. Physical posterior accuracy, MCMC throughput and ESS remain open.


In [ ]:
from pathlib import Path
import csv, hashlib, json, math, os, platform, re, signal, subprocess, sys, tempfile, time, traceback, zipfile

SCIENCE_SHA = "224aadf5b30ddb7caa585c70e7f3c4aa95841f2e"
SOURCE_URL = "https://github.com/paulnord/elisa.git"
BASE_ARGS = ["scripts/validate_gpu_reconstructed.py", "--engine", "area-thermal", "--batch", "4",
             "--repeats", "1", "--iteration-start", "0", "--levels", "0", "1",
             "--integration-level", "1", "--order", "3", "--slice-order", "4", "--depth", "1"]

def write_json(path, data):
    path.write_text(json.dumps(data, indent=2, allow_nan=False) + "\n")

def digest(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, separators=(",", ":")).encode()).hexdigest()

def select_workspace(content):
    """Reuse a matching checkout and complete environment from this Colab VM."""
    pins_name = "notebooks/observer-runtime-pins.txt"
    check_packages = """
import importlib.metadata as md, sys
from pathlib import Path
pins = dict(s.split('==') for s in Path(sys.argv[1]).read_text().splitlines() if s.strip())
assert sys.version_info[:2] == tuple(map(int, sys.argv[2:]))
assert {p: md.version(p) for p in pins} == pins
"""
    for path in sorted(content.glob("elisa-a100-*"), key=lambda p: p.stat().st_mtime, reverse=True):
        checkout, interpreter = path / "repo", path / "venv/bin/python"
        if not (checkout / ".git").is_dir() or not interpreter.is_file():
            continue
        try:
            head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=checkout, text=True, stderr=subprocess.DEVNULL)
            if head.strip() != SCIENCE_SHA:
                continue
            pins = checkout / pins_name
            if pins.read_bytes() != subprocess.check_output(["git", "show", "HEAD:" + pins_name], cwd=checkout):
                continue
            subprocess.run([interpreter, "-I", "-c", check_packages, pins, *map(str, sys.version_info[:2])],
                           check=True, capture_output=True, timeout=20)
        except (OSError, subprocess.SubprocessError):
            continue
        return path, True
    return Path(tempfile.mkdtemp(prefix="elisa-a100-", dir=content)), False

def run(name, argv, *, timeout, cwd=None, env=None):
    log = evidence / f"{name}.log"
    t0 = time.perf_counter()
    print(f"Running {name} (timeout {timeout}s) ...", flush=True)
    with log.open("w") as stream:
        process = subprocess.Popen([str(x) for x in argv], stdout=stream, stderr=subprocess.STDOUT,
                                   cwd=cwd, env=env, start_new_session=True)
        try:
            code = process.wait(timeout=timeout)
        except (subprocess.TimeoutExpired, KeyboardInterrupt):
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait()
            print(log.read_text(errors="replace")[-2500:])
            raise
    seconds = time.perf_counter() - t0
    print(f"  {name}: {seconds:.2f} s", flush=True)
    if code:
        print(log.read_text(errors="replace")[-4000:])
        raise RuntimeError(f"{name} exited {code}. See {log.name} in the evidence ZIP.")
    return seconds


### Runtime and software

The GPU model and driver are checked before installing packages. The benchmark uses the exact repository commit and its dependency pins. The isolated JAX process checks the actual CUDA device, enables FP64, and synchronizes a double-precision calculation. The benchmark gives each backend the same available CPU affinity and single-thread BLAS settings. XLA may use additional threads. The ZIP records the machine and timing boundaries.


In [ ]:
# Child processes need the driver supplied by Colab and the CUDA libraries in the venv.
def colab_driver_env(base_env):
    env = base_env.copy()
    inherited = os.environ.get("LD_LIBRARY_PATH", "").split(os.pathsep)
    known = ["/usr/lib64-nvidia", "/usr/local/nvidia/lib64", "/usr/local/nvidia/lib",
             "/usr/lib/x86_64-linux-gnu/nvidia/current", "/usr/lib/x86_64-linux-gnu"]
    driver_dirs = []
    for candidate in dict.fromkeys(inherited + known):
        directory = Path(candidate)
        if directory.is_absolute() and "stubs" not in directory.parts and (directory / "libcuda.so.1").is_file():
            driver_dirs.append(str(directory))
    # The JAX wheels already have RPATHs for their own CUDA libraries.
    if driver_dirs:
        env["LD_LIBRARY_PATH"] = os.pathsep.join(driver_dirs)
    else:
        env.pop("LD_LIBRARY_PATH", None)
    runtime["driver_search_directories"] = driver_dirs
    return env


CUDA_LOADER_PROBE = """
import ctypes, json, sys
from pathlib import Path
from jax_cuda13_plugin import _versions

driver = ctypes.CDLL('libcuda.so.1')
version = ctypes.c_int()
rc = driver.cuDriverGetVersion(ctypes.byref(version))
print('cuDriverGetVersion', rc, version.value, flush=True)
if rc: raise RuntimeError(f'GPU driver version query failed ({rc})')
rc = driver.cuInit(0)
print('cuInit', rc, flush=True)
if rc: raise RuntimeError(f'GPU driver initialization failed ({rc})')

path = Path(sys.argv[1]).resolve()
cupti = ctypes.CDLL(str(path))
direct = ctypes.c_uint()
rc = cupti.cuptiGetVersion(ctypes.byref(direct))
print('cuptiGetVersion', str(path), rc, direct.value, flush=True)
if rc: raise RuntimeError(f'CUPTI version query failed ({rc})')
actual = _versions.cupti_get_version()
assert actual == direct.value
maps = Path('/proc/self/maps').read_text().splitlines()
mapped = sorted({line.split()[-1] for line in maps if '/libcupti.so' in line})
assert {Path(name).resolve() for name in mapped} == {path}, mapped
print(json.dumps({'driver_api':version.value, 'cupti_api':actual, 'jax_cupti_build':_versions.cupti_build_version(),
                  'driver_paths':sorted({line.split()[-1] for line in maps if '/libcuda.so' in line}),
                  'cupti_paths':mapped}))
"""


def verify_cuda_runtime(env, probe, pins):
    env = colab_driver_env(env)
    bundled = next((python.parents[1] / "lib").glob("python*/site-packages/nvidia/cu13/lib/libcupti.so.13"))
    def check_loader(label, library, env):
        name = f"cuda-loader-{label}"
        run(name, [python, "-c", CUDA_LOADER_PROBE, library], env=env, timeout=30)
        return json.loads((evidence / f"{name}.log").read_text().splitlines()[-1])

    try:
        runtime["cuda_loader"] = check_loader("pinned", bundled, env)
        runtime["effective_cupti_package"] = "13.5.25"
    except RuntimeError:
        log = (evidence / "cuda-loader-pinned.log").read_text(errors="replace")
        driver_major = runtime["gpus"][0]["driver_version"].split(".")[0]
        if driver_major != "580" or "cuInit 0\n" not in log or not any(
                marker in log for marker in ("cuptiGetVersion", "libcupti.so", "cupti_get_version")):
            raise
        print("The R580 driver initialized. Retrying cuPTI with NVIDIA's CUDA 13.0 wheel (11 MB).", flush=True)
        overlay = evidence / "cupti-r580"
        lock = evidence / "cupti-r580.txt"
        lock.write_text("nvidia-cuda-cupti==13.0.85 --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8\n")
        run("cupti-r580-install", [sys.executable, "-m", "pip", "--python", str(python), "install",
                                   "--target", overlay, "--no-deps", "--only-binary=:all:", "--require-hashes",
                                   "-r", lock], timeout=120)
        library = overlay / "nvidia/cu13/lib/libcupti.so.13"
        env["LD_PRELOAD"] = os.pathsep.join(filter(None, (str(library), env.get("LD_PRELOAD"))))
        runtime["cuda_loader"] = check_loader("r580", library, env)
        runtime["effective_cupti_package"] = "13.0.85"
    print("CUDA driver and cuPTI loaded.", runtime["cuda_loader"], flush=True)
    run("jax-probe", [python, "-c", probe, pins], env={**env, "JAX_PLATFORMS": "cuda"}, timeout=90)
    return env


In [ ]:
def prepare():
    if not re.fullmatch("[0-9a-f]{40}", SCIENCE_SHA):
        raise ValueError("The scientific commit must be a full published SHA.")
    runtime["python"] = sys.version
    runtime["machine"] = platform.machine()
    runtime["host_system"] = platform.platform()
    runtime["cpu_affinity"] = sorted(os.sched_getaffinity(0))
    runtime["cpu_model"] = next((s.split(":", 1)[1].strip() for s in Path("/proc/cpuinfo").read_text().splitlines()
                                 if s.startswith("model name")), "unknown")
    quota = Path("/sys/fs/cgroup/cpu.max")
    runtime["cpu_quota"] = quota.read_text().strip() if quota.exists() else None
    fields = "name,driver_version,memory.total,pci.bus_id"
    run("hardware", ["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"], timeout=20)
    gpus = list(csv.reader((evidence / "hardware.log").read_text().splitlines(), skipinitialspace=True))
    runtime["gpus"] = [dict(zip(fields.split(","), row, strict=True)) for row in gpus]
    if len(gpus) != 1 or "A100" not in gpus[0][0]:
        raise RuntimeError(f"Select a single A100 in Colab. Detected {runtime['gpus']}")
    if int(gpus[0][1].split(".")[0]) < 580:
        raise RuntimeError(f"Driver {gpus[0][1]} is below the supported Linux CUDA 13 driver 580. Use a compatible A100 runtime.")
    if sys.version_info[:2] not in ((3, 12), (3, 13)):
        raise RuntimeError(f"Use a Colab Python 3.12 or 3.13 runtime for the pinned wheels. Found {sys.version.split()[0]}.")
    if runtime["machine"] != "x86_64":
        raise RuntimeError("This wheel set was checked on Linux x86-64.")
    print(runtime["gpus"], flush=True)
    run("gpu-inventory", ["nvidia-smi", "-L"], timeout=20)
    if reuse_setup:
        print(f"Reusing verified checkout and installed packages from {work}", flush=True)
    else:
        run("git-init", ["git", "init", str(repo)], timeout=30)
        run("source-fetch", ["git", "fetch", "--depth=1", SOURCE_URL, SCIENCE_SHA], cwd=repo, timeout=180)
        run("source-checkout", ["git", "checkout", "--detach", "FETCH_HEAD"], cwd=repo, timeout=60)
    assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip() == SCIENCE_SHA
    manifest_path = repo / "docs/ada_validation/background-m022-source-hashes.json"
    runtime["source_manifest_sha256"] = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
    assert runtime["source_manifest_sha256"] == "0d46174e9b4ec08b4d73d28c5549b39c1c63b4057931c870c0d29e112c648473"
    manifest = json.loads(manifest_path.read_text())
    for name, expected in manifest.items():
        if hashlib.sha256((repo / name).read_bytes()).hexdigest() != expected:
            raise RuntimeError(f"Scientific source checksum failed: {name}")
    runtime["scientific_files_verified"] = len(manifest)
    pins = repo / "notebooks/observer-runtime-pins.txt"
    (evidence / "requirements.txt").write_bytes(pins.read_bytes())
    if not reuse_setup:
        run("venv", [sys.executable, "-m", "venv", "--without-pip", str(work / "venv")], timeout=90)
        run("install", [sys.executable, "-m", "pip", "--python", str(python), "install", "--only-binary=:all:",
                        "--no-cache-dir", "-r", str(pins)], timeout=1200)
    config = evidence / "config.ini"
    config.write_text(f"[general]\nhome = {work / 'elisa-home'}\n[computational]\nmesh_generator = trapezoidal\n"
                      f"max_discretization_factor = 12\nnumber_of_processes = 1\nnumber_of_threads = 1\n"
                      f"[support]\nld_tables = {repo / 'tests/data/light_curves/limbdarkening'}\n"
                      f"castelli_kurucz_04_atm_tables = {repo / 'tests/data/ck04'}\n"
                      f"kurucz_93_atm_tables = {work / 'unused-k93'}\n")
    env = os.environ.copy()
    for key in ("JAX_PLATFORM_NAME", "JAX_PLATFORMS", "XLA_FLAGS"):
        env.pop(key, None)
    env.update(PYTHONPATH=os.pathsep.join(map(str, [repo / "src", repo])), ELISA_CONFIG=str(config),
               PYTHONNOUSERSITE="1", CUDA_VISIBLE_DEVICES="0", JAX_ENABLE_X64="True",
               XLA_PYTHON_CLIENT_PREALLOCATE="false", XDG_CACHE_HOME=str(attempt / "cache"),
               OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", NUMEXPR_NUM_THREADS="1")
    probe = """
import importlib.metadata as md, json, sys
from pathlib import Path
import jax, numpy as np
pins = dict(s.split('==') for s in Path(sys.argv[1]).read_text().splitlines() if s.strip())
assert {p: md.version(p) for p in pins} == pins
gpu = jax.devices('gpu')[0]
a = jax.device_put(np.array([1.5, 2.25], dtype=np.float64), gpu)
out = (a*a).block_until_ready()
assert 'A100' in gpu.device_kind and jax.config.x64_enabled and out.dtype == np.float64 and out.device.platform == 'gpu'
np.testing.assert_array_equal(np.asarray(out), [2.25, 5.0625])
print(json.dumps(dict(device=gpu.device_kind, device_description=str(gpu), x64=True, packages=pins)))
"""
    env = verify_cuda_runtime(env, probe, pins)
    runtime["jax_probe"] = json.loads((evidence / "jax-probe.log").read_text().splitlines()[-1])
    runtime["thread_env"] = {k: env[k] for k in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS")}
    return env


### Matched batches and accuracy checks

The benchmark executes the A100 path first so a compiler or GPU failure is identified early. Each child process includes imports, setup, compilation, two physically varying batches, transfers, synchronization and returned host data. Device work has a 270-second process limit. The faster local CPU or GPU can differ across hosts. Compare the reported whole-process ratio and warm ratio separately. No projections from the A100’s advertised bandwidth are used.


In [ ]:
def benchmark(env):
    profiles, times = {}, {}
    for backend, limit in (("gpu", 270), ("reference", 480), ("cpu", 600)):
        path = evidence / f"complete-{backend}.json"
        args = [python, *BASE_ARGS, "--backend", backend, "--output", path]
        child_env = env.copy()
        if backend == "gpu":
            args.append("--verified-bounded-gpu-window")
        else:
            child_env["JAX_PLATFORMS"] = "cpu"
        seconds = run(backend, args, env=child_env, cwd=repo, timeout=limit)
        p = json.loads(path.read_text())
        assert p["backend"] == backend and p["batch"] == 4 and len(p["samples"]) == 2
        assert all(len(s["rows"]) == 4 for s in p["samples"])
        profiles[backend] = p
        first, warm = [s["enclosing_complete_batch_seconds"] for s in p["samples"]]
        times[backend] = dict(first_batch=first, warm_batch=warm, process=seconds,
                              input_setup=p["fixed_input_setup_seconds"], constructor=p["constructor_seconds"])
        print(f"  First / warm complete B4: {first:.3f} / {warm:.3f} s")
    return profiles, times

def same_inputs(a, b, *, cross_host=False):
    for key in ("source_revision", "source_sha256", "engine", "batch", "jd", "levels", "integration_level", "depth",
                "slice_order", "order", "uncertainties", "reducers", "physical_constants",
                "iteration_start", "irradiation", "observations_irradiated"):
        if a[key] != b[key]:
            raise ValueError(f"Different fixed inputs or protocol: {key}")
    if cross_host:
        assert a["observations"].keys() == b["observations"].keys()
        for band, values in a["observations"].items():
            close(b["observations"][band], values, 3e-13, 2e-12, "cross-host synthetic observations")
    else:
        assert a["observations"] == b["observations"]

def close(arr, ref, rtol, atol, label):
    if isinstance(ref, list):
        return max((close(x,y,rtol,atol,label) for x,y in zip(arr,ref,strict=True)), default=0.)
    if not (math.isfinite(arr) and math.isfinite(ref)) or abs(arr-ref) > atol + rtol*abs(ref):
        raise ValueError(f"{label}: {arr} versus {ref} (rtol={rtol}, atol={atol})")
    return abs(arr-ref)

def check_parity(ref, test):
    same_inputs(ref, test)
    peak = peak_logp = 0.
    decisions = []
    for a, b in zip(ref["samples"], test["samples"], strict=True):
        for r, s in zip(a["rows"], b["rows"], strict=True):
            for k in ("physical", "phase_records", "finite_metadata", "accuracy_abort"):
                if r[k] != s[k]:
                    raise ValueError(f"{test['backend']}: {k} differs from the CPU finite-model reference")
            assert r["curves"].keys() == s["curves"].keys()
            for model, variants in r["curves"].items():
                assert variants.keys() == s["curves"][model].keys()
                for attr, bands in variants.items():
                    assert bands.keys() == s["curves"][model][attr].keys()
                    for band, expected in bands.items():
                        error = close(s["curves"][model][attr][band], expected, 3e-13,
                                      2e-12 if attr == "fluxes" else 0., attr)
                        if attr == "fluxes":
                            peak = max(peak, error)
            assert s["prediction"].keys() == r["prediction"].keys()
            for band, expected in r["prediction"].items():
                peak = max(peak, close(s["prediction"][band], expected, 3e-13, 2e-12, "prediction"))
            peak_logp = max(peak_logp, close(s["posterior"], r["posterior"], 0., 1e-6, "log posterior"))
            controls = [row["accuracy_control"] for row in (r,s)]
            reports = [c["nominal_report"] for c in controls]
            bounds = [v["finite_log_likelihood_absolute_bound"] for v in reports]
            close(bounds[1], bounds[0], 0., 1e-6, "nominal field log-likelihood bound")
            for row, c, v in zip((r,s),controls,reports,strict=True):
                assert c["max_distance_sigma"] == 0.1 and c["max_log_likelihood"] == .01
                assert c["within_finite_budget"] != row["accuracy_abort"]
                assert not c["nuisance_noise_used_for_gate"] and not c["continuum_certified"]
            decisions.append({"accuracy_stop":s["accuracy_abort"],"nominal_field_bound":bounds[1],
                              "nominal_distance":reports[1]["nominal_component_bound"]["distance_sigma"]})
    return {"max_abs_normalized_flux_or_prediction":peak, "max_abs_log_posterior":peak_logp,
            "physical_inputs_and_adaptive_traces_match":True, "finite_field_checks":decisions}


In [ ]:
def summarize(profiles, times):
    result = {"comparison_workload":"Two finite area-thermal curves per proposal. B4, one warm repeat.",
              "within_host_parity":{}, "timing_seconds":times, "physical_MCMC_qualified":False}
    for backend in ("gpu", "cpu"):
        result["within_host_parity"][backend] = check_parity(profiles["reference"], profiles[backend])
    result["local_serial_reference_over_GPU"] = {k:times["reference"][k]/times["gpu"][k] for k in ("warm_batch","process")}
    archived = json.loads((repo / "docs/ada_validation/background-m022.json").read_text())
    cross = {"same_protocol_and_inputs":False, "compared_source_windows":{}, "adaptive_trace_hash_matches":{}}
    try:
        for backend, window in (("reference","211"), ("gpu","210"), ("cpu","212")):
            original, current = archived["profiles"][window], profiles[backend]
            same_inputs(original, current, cross_host=True)
            trace_matches = []
            for a,b in zip(original["samples"], current["samples"],strict=True):
                for r,s in zip(a["rows"],b["rows"],strict=True):
                    assert r["physical"] == s["physical"] and r["accuracy_abort"] == s["accuracy_abort"]
                    assert r["finite_metadata"].keys() == s["finite_metadata"].keys()
                    for model, meta in r["finite_metadata"].items():
                        new = s["finite_metadata"][model]
                        for k in ("rule", "field_levels", "additional_light"):
                            assert meta[k] == new[k], k
                        close(new["phases"], meta["phases"], 3e-13, 2e-15, "cross-host observation phases")
                    records = s["phase_records"]
                    assert len(records) == r["unique_processed_positions"]
                    for component, n in r["accepted_packet_nodes_by_component"].items():
                        assert sum(item["components"][component]["nodes"] for item in records) == n, component
                    trace_matches.append(r["full_adaptive_trace_sha256"] == digest(records))
                    assert r["prediction"].keys() == s["prediction"].keys()
                    for band, values in r["prediction"].items():
                        close(s["prediction"][band],values,3e-13,2e-12,"cross-host prediction")
                    close(s["posterior"],r["posterior"],0.,1e-6,"cross-host log posterior")
                    report = s["accuracy_control"]["nominal_report"]
                    close(report["finite_log_likelihood_absolute_bound"],r["nominal_log_likelihood_enclosure"],
                          0.,1e-6,"cross-host finite log-likelihood bound")
                    close(report["nominal_component_bound"]["distance_sigma"],r["nominal_D_enclosure"],
                          0.,1e-6,"cross-host nominal distance")
            cross["compared_source_windows"][backend] = int(window)
            cross["adaptive_trace_hash_matches"][backend] = sum(trace_matches)
        original = archived["profiles"]["210"]
        cross.update(same_protocol_and_inputs=True,
                     RTX3080Ti_warm_seconds=original["samples"][1]["enclosing_complete_batch_seconds"],
                     RTX3080Ti_process_seconds=original["complete_process_seconds"])
        cross["RTX3080Ti_over_A100_warm"] = cross["RTX3080Ti_warm_seconds"]/times["gpu"]["warm_batch"]
        cross["RTX3080Ti_over_A100_process"] = cross["RTX3080Ti_process_seconds"]/times["gpu"]["process"]
    except (ValueError, AssertionError) as exc:
        cross["status"] = f"Cross-host input, adaptive node count or output mismatch. Inspect raw profiles. {exc}"
    result["cross_host"] = cross
    return result

from google.colab import files
started = time.perf_counter()
work, reuse_setup = select_workspace(Path("/content"))
repo = work / "repo"
python = work / "venv/bin/python"
attempt = Path(tempfile.mkdtemp(prefix="attempt-", dir=work))
evidence = attempt / "evidence"
evidence.mkdir()
runtime = {"science_commit": SCIENCE_SHA, "argv_common":BASE_ARGS, "reused_setup":reuse_setup}
result = {"state":"started"}
try:
    env = prepare()
    runtime["setup_wall_seconds"] = time.perf_counter()-started
    profiles, times = benchmark(env)
    result = summarize(profiles,times)
    result["effective_cupti_package"] = runtime["effective_cupti_package"]
    result["cross_host"]["cupti_matches_RTX_pin"] = runtime["effective_cupti_package"] == "13.5.25"
    result["state"] = "complete"
    print(json.dumps({key:result[key] for key in ("timing_seconds","local_serial_reference_over_GPU","cross_host")}, indent=2))
    for backend, p in result["within_host_parity"].items():
        stops = sum(c["accuracy_stop"] for c in p["finite_field_checks"])
        print(f"{backend}: parity passed, {stops}/8 finite-field accuracy stops. Physical MCMC remains unqualified.")
except BaseException as exc:
    result = {"state":"interrupted" if isinstance(exc,KeyboardInterrupt) else "failed",
              "error":f"{type(exc).__name__}: {exc}"}
    (evidence / "exception.txt").write_text(traceback.format_exc())
    raise
finally:
    runtime["total_wall_seconds_before_packaging"] = time.perf_counter() - started
    write_json(evidence/"runtime.json", runtime)
    write_json(evidence/"summary.json", result)
    archive = attempt / "elisa-a100-comparison.zip"
    with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in sorted(evidence.glob("*")):
            if path.is_file():
                bundle.write(path, path.name)
    print("Evidence ZIP:", archive, flush=True)
    try:
        files.download(str(archive))
    except Exception as exc:
        print(f"Browser download unavailable ({exc}). Use the Colab Files panel to download {archive}.")
